# Estudio de ablación — `75mil_tarjetas_enero_junio_2025`

Se parte de la **configuración óptima** (trial #57 de `mdlf_junio2025_v2_line`, Top-1 proxy = 0.7070 con 7 épocas)  
y se desactivan o modifican uno a uno los componentes de interés para medir su contribución real.

### Experimentos

| # | Nombre | Cambio respecto al baseline |
|---|---|---|
| 0 | `baseline` | Configuración completa (trial #57) |
| 1 | `sin_neighborhood` | `include_neighborhood_crowd=False` |
| 2 | `sin_prox` | `include_prox=False` |
| 3 | `sgd` | `optimizer='sgd'` (mismo `lr` — comparación orientativa) |
| 4 | `sin_neighborhood_sin_prox` | `include_neighborhood_crowd=False` + `include_prox=False` |

Todos los experimentos usan **30 épocas** (entrenamiento completo, no proxy).

In [ ]:
import os, sys
from pathlib import Path
from datetime import date

_cubo_dir = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo').resolve()
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF').resolve()
_estudios = _cubo_dir.parent

os.chdir(_cubo_dir)
for _p in [str(_cubo_dir), str(_mdlf_dir), str(_estudios)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pipeline_mdlf import PipelineMDLF

print(f'CWD      → {os.getcwd()}')
print(f'MDLF dir → {_mdlf_dir}')

In [ ]:
# ── Constantes ────────────────────────────────────────────────────────────────

ALMACEN    = '75mil_tarjetas_enero_junio_2025'
SPLIT_DATE = '2025-06-01'
N_EPOCHS   = 10

# LINE entrenado una sola vez antes del bucle de Optuna v2 75mil
LINE_PATH = _mdlf_dir / 'runs' / 'line_fixed_optuna.parquet'
assert LINE_PATH.exists(), f'LINE parquet no encontrado: {LINE_PATH}'
print(f'LINE parquet : {LINE_PATH.name}  ✓')

# Festivos enero–junio 2025
HOLIDAY_DATES = frozenset([
    date(2025,  1,  1),
    date(2025,  1,  6),
    date(2025,  4, 17),
    date(2025,  4, 18),
    date(2025,  5,  1),
    date(2025,  5,  2),
    date(2025,  5, 15),
])

_descomp = _estudios / 'descomprimido'
TTP_PATHS = [str(_descomp / f'TTP_PERIODO_0{i}_2025.csv') for i in range(1, 7)]

OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero',
    'validaciones_8_9_febrero',
    'validaciones_10_16_febrero',
    'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

# ── Hiperparámetros óptimos — trial #57, mdlf_junio2025_v2_line ───────────────
# top1_proxy (7 épocas) = 0.7070
BEST_PARAMS = {
    # LDA
    'n_topics_cercanias'    : 48,
    'lda_alpha'             : 0.2122,
    'lda_beta_none'         : False,
    'lda_beta'              : 0.0029,
    'lda_log_counts'        : True,
    'lda_min_df'            : 5,
    'lda_min_station_trips' : 0,
    # LDA fijos
    'lda_use_periods'       : False,
    'lda_token_granularity' : 'temporal',
    # CNN
    'n_heads'      : 2,
    'd_t'          : 8,
    'cnn_channels' : 32,
    'kernel_size'  : 3,
    # Entrenamiento
    'optimizer'    : 'adamw',
    'lr'           : 0.0056,
    'weight_decay' : 0.0007,
    'lr_scheduler' : True,
    'n_weeks'      : 5,
    # Extensiones
    'include_prox'               : True,
    'include_neighborhood_crowd' : True,
    'td_mode'                    : 'context',
    'norm'                       : 'none',
}

print(f'Almacén    : {ALMACEN}')
print(f'Split date : {SPLIT_DATE}')
print(f'N_epochs   : {N_EPOCHS}')

In [ ]:
# ── Helper ────────────────────────────────────────────────────────────────────

def run_ablation(name: str, **overrides) -> dict:
    """
    Ejecuta PipelineMDLF con BEST_PARAMS + overrides.
    Devuelve dict con name, top1, top5.
    """
    params = {**BEST_PARAMS, **overrides}

    lda_beta = None if params.pop('lda_beta_none', True) else params.pop('lda_beta', None)
    norm_val = params.pop('norm', 'none')
    norm = None if norm_val == 'none' else norm_val

    _skip = {'lda_beta', 'lda_beta_none', 'norm'}
    pipe_kwargs = {k: v for k, v in params.items() if k not in _skip}

    pipe = PipelineMDLF(
        nombre_almacen=ALMACEN,
        ttp_paths=TTP_PATHS,
        od_matrices_subdirs=OD_MATRICES_SUBDIRS,
        line_ap_path=LINE_PATH,
        split_date=SPLIT_DATE,
        holiday_dates=HOLIDAY_DATES,
        lda_beta=lda_beta,
        norm=norm,
        n_epochs=N_EPOCHS,
        random_state=42,
        **pipe_kwargs,
    )

    pipe.run(run_name=f'ablacion_75mil_{name}')
    result = {'name': name, 'top1': pipe.test_accuracy, 'top5': pipe.test_accuracy5}
    print(f'  [{name}]  Top-1={result["top1"]:.4f}  Top-5={result["top5"]:.4f}')
    return result

In [ ]:
# ── Exp 0: Baseline (resultado ya obtenido — 30 épocas) ──────────────────────
# Top-1=0.7025  Top-5=0.8761  (log: ablacion_75mil_baseline_20260531_212422.log)
# No se re-ejecuta para no perder tiempo. Los experimentos siguientes usan 10 épocas.
# NOTA: la comparación de deltas es orientativa (baseline=30 ép, resto=10 ép).

results = []
results.append({'name': 'baseline (30 épocas)', 'top1': 0.7025, 'top5': 0.8761})
print('  [baseline]  Top-1=0.7025  Top-5=0.8761  (cargado desde log, 30 épocas)')

In [ ]:
# ── Exp 1: Sin neighborhood_crowd ────────────────────────────────────────────
# Se desactivan las features de demanda colectiva de estaciones vecinas
# con la misma parada de residencia.
# Si delta_top1 < 0: esas features aportan señal real sobre el destino.

results.append(run_ablation('sin_neighborhood', include_neighborhood_crowd=False))

In [ ]:
# ── Exp 2: Sin include_prox ───────────────────────────────────────────────────
# Se eliminan las features de proximidad espacial (validaciones en medios
# complementarios próximos en tiempo a la subida).
# Si delta_top1 < 0: la señal de modos adyacentes ayuda a predecir el destino.

results.append(run_ablation('sin_prox', include_prox=False))

In [ ]:
# ── Exp 3: SGD en lugar de AdamW ─────────────────────────────────────────────
# Se sustituye AdamW por SGD manteniendo el mismo lr (0.0056).
# La comparación no es del todo justa (SGD puede necesitar lr distinto),
# pero mide el impacto del optimizador sin reoptimizar.

results.append(run_ablation('sgd', optimizer='sgd'))

In [ ]:
# ── Exp 4: Sin neighborhood_crowd + sin include_prox ─────────────────────────
# Desactiva simultáneamente ambos módulos de señal contextual externa.
# Permite distinguir si sus contribuciones son independientes o solapadas.

results.append(run_ablation('sin_neighborhood_sin_prox',
                             include_neighborhood_crowd=False,
                             include_prox=False))

---
## Experimento adicional: configuración del paper original (MDLF, Zhao et al. 2022)

Se replican los hiperparámetros del paper original (Sección VI.B) para comparación directa.
Los módulos `include_prox` e `include_neighborhood_crowd` no existen en el paper y se desactivan.

| Parámetro | Valor paper | Fuente |
|---|---|---|
| `n_topics_cercanias` K | `128` | D2=128 (region function dim) |
| `lda_alpha` | `None` (simétrico) | No especificado en paper |
| `lda_beta` | `None` (simétrico) | No especificado en paper |
| `lda_log_counts` | `False` | No mencionado en paper |
| LINE dims | `128` | D1=128 (region remoteness dim), `line_fixed_optuna.parquet` ya es dim=64→128 |
| `optimizer` | `sgd` | SGD con momentum, lr=0.01 (Sección VI.B) |
| `lr` | `0.01` | Rango ajustado: 0.001–0.01 |
| `weight_decay` | `0.0` | No mencionado |
| `n_heads` H | `4` | Best setting Sección VI.B |
| `cnn_channels` C | `16` | Best setting Sección VI.B |
| `n_weeks` W | `8` | Definición 2 del paper |
| `include_prox` | `False` | No existe en el paper |
| `include_neighborhood_crowd` | `False` | No existe en el paper |


In [ ]:
# ── Exp 5: Configuración del paper original ───────────────────────────────────
# Replica los hiperparámetros de Zhao et al. (2022) — IEEE T-ITS.
#
# CNN/Train : H=4, C=16, SGD lr=0.01, n_weeks=8
# LINE      : dim=64 → 128 dims (FO+SO), igual que line_fixed_optuna.parquet
# LDA       : K=128, alpha=None (simétrico), beta=None (simétrico), log_counts=False
# Sin prox ni neighborhood (no existen en el paper)

results.append(run_ablation(
    'paper_original',
    # CNN + entrenamiento
    optimizer='sgd',
    lr=0.01,
    weight_decay=0.0,
    n_heads=4,
    cnn_channels=16,
    n_weeks=8,
    # LDA — paper: K=128, alpha y beta simétricos (None), sin log_counts
    n_topics_cercanias=128,
    lda_alpha=None,
    lda_beta_none=True,      # beta=None
    lda_log_counts=False,
    lda_min_df=1,
    lda_min_station_trips=0,
    # LINE: line_fixed_optuna.parquet ya tiene dim=64 → 128 dims, igual que el paper
    # Sin extensiones propias
    include_prox=False,
    include_neighborhood_crowd=False,
))

In [ ]:
# ── Tabla de resultados completa (ablación + paper original) ──────────────────

import pandas as pd

df_results = pd.DataFrame(results)

baseline_top1 = df_results.loc[df_results['name'] == 'baseline', 'top1'].values[0]
baseline_top5 = df_results.loc[df_results['name'] == 'baseline', 'top5'].values[0]

df_results['delta_top1'] = df_results['top1'] - baseline_top1
df_results['delta_top5'] = df_results['top5'] - baseline_top5

df_results = df_results.sort_values('top1', ascending=False).reset_index(drop=True)

print(f'Baseline (trial #57) → Top-1={baseline_top1:.4f}  Top-5={baseline_top5:.4f}')
print()

display(
    df_results.style.format({
        'top1'      : '{:.4f}',
        'top5'      : '{:.4f}',
        'delta_top1': '{:+.4f}',
        'delta_top5': '{:+.4f}',
    }).background_gradient(
        subset=['delta_top1'], cmap='RdYlGn', vmin=-0.05, vmax=0.01
    )
)

In [ ]:
# ── Paper original vs. mejor combinación DB (trial #57) ──────────────────────

import pandas as pd

df = pd.DataFrame(results)

paper = df[df['name'] == 'paper_original'].iloc[0]
best  = df[df['name'] == 'baseline'].iloc[0]

delta_top1 = best['top1'] - paper['top1']
delta_top5 = best['top5'] - paper['top5']

summary = pd.DataFrame([
    {
        'Configuración'  : 'Paper original (Zhao et al. 2022)',
        'Top-1'          : paper['top1'],
        'Top-5'          : paper['top5'],
        'Δ Top-1 vs best': 0.0,
        'Δ Top-5 vs best': 0.0,
    },
    {
        'Configuración'  : 'Mejor config DB (trial #57, Optuna)',
        'Top-1'          : best['top1'],
        'Top-5'          : best['top5'],
        'Δ Top-1 vs best': delta_top1,
        'Δ Top-5 vs best': delta_top5,
    },
])

print('── Paper original vs. mejor configuración Optuna ────────────────────────')
print(f"  Paper original  →  Top-1: {paper['top1']:.4f}  |  Top-5: {paper['top5']:.4f}")
print(f"  Mejor DB        →  Top-1: {best['top1']:.4f}  |  Top-5: {best['top5']:.4f}")
print(f"  Mejora Top-1    →  {delta_top1:+.4f}  ({delta_top1/paper['top1']*100:+.2f}%)")
print(f"  Mejora Top-5    →  {delta_top5:+.4f}  ({delta_top5/paper['top5']*100:+.2f}%)")
print()

display(
    summary.style.format({
        'Top-1'          : '{:.4f}',
        'Top-5'          : '{:.4f}',
        'Δ Top-1 vs best': '{:+.4f}',
        'Δ Top-5 vs best': '{:+.4f}',
    }).background_gradient(
        subset=['Top-1'], cmap='RdYlGn', vmin=0.60, vmax=0.75
    ).set_caption('Comparación paper original vs. configuración optimizada (Optuna, 75mil dataset, 30 épocas)')
)